# SFT → DPO → Eval on Colab Pro A100

Run this AFTER `colab_pretrain_60m.ipynb` produced `checkpoints/base_60m/final.pt`.

End-to-end pipeline:
1. Build SFT dataset from TinyStoriesInstruct
2. Train SFT (~1 hour A100)
3. Sample candidate preference pairs from SFT model (~1 hour)
4. Label preferences with Qwen-2.5-3B-Instruct judge (~1 hour)
5. Train DPO (~2 hours)
6. Full eval matrix → results.md (~40 min)

**Total time on A100:** ~6 hours. Set this overnight.

## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/transformer-lm'
for d in ['checkpoints/sft', 'checkpoints/dpo', 'data', 'logs']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)

Mounted at /content/drive


In [2]:
%cd /content
!git clone https://github.com/Qwerty0826/char-level-transformer-lm.git || (cd char-level-transformer-lm && git pull)
%cd /content/char-level-transformer-lm
!pip install -e . -q
!pip install datasets transformers bitsandbytes accelerate -q

!rm -rf data checkpoints logs
!ln -sf {DRIVE_ROOT}/data        data
!ln -sf {DRIVE_ROOT}/checkpoints checkpoints
!ln -sf {DRIVE_ROOT}/logs        logs
!ls -la checkpoints/

/content
Cloning into 'char-level-transformer-lm'...
remote: Enumerating objects: 308, done.
remote: Counting objects: 100% (308/308), done.
remote: Compressing objects: 100% (199/199), done.
remote: Total 308 (delta 150), reused 244 (delta 94), pack-reused 0 (from 0)
Receiving objects: 100% (308/308), 7.60 MiB | 13.48 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/content/char-level-transformer-lm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cs336-basics (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.4 MB/s eta 0:00:00:00:0100:01
total 24
drwx------ 2 root root 4096 May 25 18:43 base_60m
drwx------ 2 root root 4096 May 26 13:34 dpo
drwx------ 2 root root 4096 May 27 13:59 dpo_v2
drwx------ 2 root root 4096 May 26 

In [10]:
!pip install gradio -q

!python scripts/playground.py \
    --checkpoint     checkpoints/base_60m/final.pt \
    --checkpoint_sft checkpoints/sft/final.pt \
    --checkpoint_dpo checkpoints/dpo/final.pt \
    --vocab  data/tinystories_v2_vocab.json \
    --merges data/tinystories_v2_merges.txt \
    --dtype bfloat16 \
    --share

[playground] Loaded BASE from checkpoints/base_60m/final.pt  (step 20,000)
[playground] Loaded SFT from checkpoints/sft/final.pt  (step 2,000)
[playground] Loaded DPO from checkpoints/dpo/final.pt  (step 400)
[playground] 53,261,440-param model on cuda  (3 checkpoint(s) loaded)
/content/char-level-transformer-lm/scripts/playground.py:315: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Transformer LM Playground", theme=theme) as app:
/content/char-level-transformer-lm/scripts/playground.py:373: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  stop_btn.click(fn=None, inputs=None, outputs=None, cancels=[gen_event])
* Running on local URL:  http://127.0.0.1:7860
* Running o

In [3]:
# Verify the base pretrain finished and produced a final.pt.
assert os.path.exists('checkpoints/base_60m/final.pt'), \
    'Base pretrain checkpoint missing. Run colab_pretrain_60m.ipynb first.'
print('Base checkpoint:', os.path.getsize('checkpoints/base_60m/final.pt') / 1e6, 'MB')

Base checkpoint: 319.642595 MB


In [3]:
!cd /content/char-level-transformer-lm && git pull


Already up to date.


In [5]:
!ls -la checkpoints/sft/final.pt data/pref_labeled.jsonl
!wc -l data/pref_labeled.jsonl


-rw------- 1 root root 319642595 May 26 18:58 checkpoints/sft/final.pt
-rw------- 1 root root    301072 May 27 05:08 data/pref_labeled.jsonl
178 data/pref_labeled.jsonl


## 1. Build the SFT dataset (~5 min)

In [9]:
!python scripts/build_sft_dataset.py \
    --vocab  data/tinystories_v2_vocab.json \
    --merges data/tinystories_v2_merges.txt \
    --output data/tinystories_v2_sft.pt \
    --max_length 512 --max_examples 40000

Loaded tokenizer: vocab_size=16000, pad_id=256

Loading roneneldan/TinyStoriesInstruct ...
  21,755,681 raw examples

Formatting (max_length=512, max_examples=40000) ...
  Saw 42,073 story blocks; kept 40,000 examples in 78.5s
  Skipped: parse_fail=2,073, too_short=0

Collating + padding to 512 ...
  Train: 39,200 examples
  Val:   800 examples
  Avg response tokens per example (train): 211.7

Saved data/tinystories_v2_sft.pt (429.2 MB)


## 2. SFT training (~1 hour A100)

In [12]:
# ── OVERNIGHT PIPELINE: SFT → sanity → build prefs → label prefs ──
# Click Run on this ONE cell. Stops cleanly BEFORE DPO training.
# Expected total on L4: ~6-8 hours (SFT ~45min, build prefs ~1.5-2h, label ~3-5h).
# Each step is fail-fast: if one crashes, subsequent steps are skipped so you
# wake up to a clear traceback instead of a corrupted half-run pipeline.

import subprocess, sys, time, os, gc
from pathlib import Path

t_start = time.time()

def run_step(cmd, label):
    print(f"\n{'='*70}\n  {label}  [started at {time.strftime('%H:%M:%S')}]\n{'='*70}\n", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, shell=True)
    mins = (time.time() - t0) / 60
    if r.returncode != 0:
        raise SystemExit(f"\nFAILED: {label} (exit {r.returncode}) after {mins:.1f} min")
    print(f"\n  ✓ {label} OK in {mins:.1f} min", flush=True)

# ── 1/4  SFT training ──
run_step(
    "python scripts/train_sft.py "
    "--base_checkpoint checkpoints/base_60m/final.pt "
    "--sft_data        data/tinystories_v2_sft.pt "
    "--checkpoint_dir  checkpoints/sft "
    "--total_steps 2000 --batch_size 32 "
    "--lr_max 3e-5 --warmup_steps 100 "
    "--device cuda --dtype bfloat16 --compile",
    "1/4  SFT training"
)
assert Path("checkpoints/sft/final.pt").exists(), "SFT final.pt missing after training!"
sft_size_mb = Path("checkpoints/sft/final.pt").stat().st_size / 1e6
print(f"  checkpoints/sft/final.pt: {sft_size_mb:.1f} MB")

# ── 2/4  Sanity check (chat template respected?) ──
print(f"\n{'='*70}\n  2/4  SFT sanity check  [started at {time.strftime('%H:%M:%S')}]\n{'='*70}\n", flush=True)
def _sanity():
    import torch
    from cs336_basics.model import TransformerLM
    from cs336_basics.tokenizer import Tokenizer
    from cs336_basics.data_sft import USER_TAG, ASSISTANT_TAG, EOT

    tok = Tokenizer.from_files(
        "data/tinystories_v2_vocab.json",
        "data/tinystories_v2_merges.txt",
        special_tokens=["<|endoftext|>", "<|user|>", "<|assistant|>", "<|system|>"],
    )
    model = TransformerLM(
        vocab_size=16000, context_length=512, d_model=640, num_layers=10,
        num_heads=10, num_kv_heads=2, d_ff=1728, device="cuda", dtype=torch.bfloat16,
    )
    ckpt = torch.load("checkpoints/sft/final.pt", map_location="cpu")
    state = ckpt.get("model_state_dict", ckpt)
    if any(k.startswith("_orig_mod.") for k in state):
        state = {k.removeprefix("_orig_mod."): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    model.eval()

    eot = tok.encode(EOT)[0]
    prompt = USER_TAG + "Write a story about a fox who learns to share." + EOT + ASSISTANT_TAG
    ids = torch.tensor([tok.encode(prompt)], device="cuda")
    out = model.generate(ids, max_new_tokens=200, temperature=0.8, top_p=0.95, eos_id=eot)
    print(tok.decode(out[0].tolist()))
    # Release VRAM so the next subprocess has a clean GPU.
    del model, ckpt, state, ids, out, tok
_sanity()
gc.collect()
import torch as _t; _t.cuda.empty_cache()
print("\n  ✓ Sanity OK", flush=True)

# ── 3/4  Build candidate preference pairs ──
run_step(
    "python scripts/build_preference_dataset.py "
    "--sft_checkpoint checkpoints/sft/final.pt "
    "--vocab          data/tinystories_v2_vocab.json "
    "--merges         data/tinystories_v2_merges.txt "
    "--output         data/pref_candidates.jsonl "
    "--num_prompts 1000 --max_new_tokens 200 "
    "--device cuda --dtype bfloat16",
    "3/4  Build preference pairs"
)
cand_path = Path("data/pref_candidates.jsonl")
assert cand_path.exists() and cand_path.stat().st_size > 0, "pref_candidates.jsonl is empty!"
n_cand = sum(1 for _ in cand_path.open())
print(f"  pref_candidates.jsonl: {n_cand:,} pairs ({cand_path.stat().st_size/1024:.1f} KB)")
assert n_cand >= 50, f"Only {n_cand} candidate pairs — too few. Inspect the SFT model output."

# ── 4/4  Label preferences with Qwen judge ──
run_step(
    "python scripts/label_preferences.py "
    "--input  data/pref_candidates.jsonl "
    "--output data/pref_labeled.jsonl "
    "--judge_model Qwen/Qwen2.5-3B-Instruct "
    "--device cuda --load_in_4bit",
    "4/4  Label preferences"
)
lab_path = Path("data/pref_labeled.jsonl")
assert lab_path.exists() and lab_path.stat().st_size > 0, "pref_labeled.jsonl is empty!"
n_lab = sum(1 for _ in lab_path.open())
print(f"  pref_labeled.jsonl: {n_lab:,} pairs ({lab_path.stat().st_size/1024:.1f} KB)")
assert n_lab >= 30, f"Only {n_lab} labeled pairs — judge dropped too many. DPO will be weak."

total_h = (time.time() - t_start) / 3600
print(f"\n{'='*70}\n  ALL STEPS COMPLETE  ({total_h:.2f} hours total)\n{'='*70}")
print("Ready to run cell with DPO training (preferably on a fresh A100 runtime).")



  1/4  SFT training  [started at 18:45:13]


  ✓ 1/4  SFT training OK in 13.6 min
  checkpoints/sft/final.pt: 319.6 MB

  2/4  SFT sanity check  [started at 18:58:46]

<|user|>Write a story about a fox who learns to share.<|endoftext|><|assistant|>Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine. One day, she found a shiny coin in the grass. She was so happy! She ran to show her mommy, but when they got to eat it, something unexpected happened. 
Lily felt dizzy from all the count of thewen. She asked her mommy if they could have a bite. Her mommy said yes and they both took a bite of the coin. 
Lily was very happy. She knew that the coin was very important and had worked hard to keep her safe. From that day on, Lily made sure to always remain kind to the sun and to always be kind to everyone she met.<|endoftext|>

  ✓ Sanity OK

  3/4  Build preference pairs  [started at 18:58:53]


  ✓ 3/4  Build preference pairs OK in 130.9 min
  pref_c

SystemExit: 
FAILED: 4/4  Label preferences (exit 1) after 0.6 min

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
%tb


In [ ]:
import subprocess
result = subprocess.run(['ps', 'aux'], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'train_sft' in line:
        print(line)


In [5]:
from datasets import load_dataset
ds = load_dataset("roneneldan/TinyStoriesInstruct", split="train")
print("Columns:", ds.column_names)
print("\n--- First example ---")
ex = ds[0]
for k, v in ex.items():
    preview = str(v)[:300]
    print(f"[{k}]: {preview}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


TinyStories-Instruct-train.txt:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

TinyStories-Instruct-valid.txt:   0%|          | 0.00/26.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21755681 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/218380 [00:00<?, ? examples/s]

Columns: ['text']

--- First example ---
[text]: Features: Dialogue


In [7]:
for i in range(40):
    text = ds[i]['text']
    print(f'[{i:3d}] {repr(text)[:150]}')


[  0] 'Features: Dialogue'
[  1] 'Words: quit, oak, gloomy'
[  2] 'Summary: Sara and Ben were playing in the park, but Sara wanted to go home because it was cold and dark. Ben convinced her to stay and play, but even
[  3] 'Story: '
[  4] ''
[  5] 'Sara and Ben were playing in the park. They liked to climb the big oak tree and pretend they were birds. They made nests with leaves and twigs and sa
[  6] 'But today, the sky was gloomy and the wind was cold. Sara felt sad and cold. She wanted to go home and have some hot cocoa.'
[  7] '"Ben, I want to quit," she said. "It\'s too cold and dark. Let\'s go home."'
[  8] 'Ben looked at Sara and frowned. He liked the oak tree and the park. He wanted to stay and play.'
[  9] '"No, Sara, don\'t quit," he said. "It\'s fun here. Look, there\'s a squirrel. Let\'s chase it."'
[ 10] "Sara shook her head. She didn't want to chase the squirrel. She wanted to go home and have some hot cocoa."
[ 11] '"Please, Ben, let\'s go home," she said. "We can play h

In [ ]:
# Manual sanity check: does the SFT model respect the chat template?
import torch
from cs336_basics.model import TransformerLM
from cs336_basics.tokenizer import Tokenizer
from cs336_basics.data_sft import USER_TAG, ASSISTANT_TAG, EOT

tok = Tokenizer.from_files(
    'data/tinystories_v2_vocab.json',
    'data/tinystories_v2_merges.txt',
    special_tokens=['<|endoftext|>', '<|user|>', '<|assistant|>', '<|system|>'],
)
model = TransformerLM(
    vocab_size=16000, context_length=512, d_model=640, num_layers=10,
    num_heads=10, num_kv_heads=2, d_ff=1728, device='cuda', dtype=torch.bfloat16,
)
ckpt = torch.load('checkpoints/sft/final.pt', map_location='cpu')
state = ckpt.get('model_state_dict', ckpt)
if any(k.startswith('_orig_mod.') for k in state):
    state = {k.removeprefix('_orig_mod.'): v for k, v in state.items()}
model.load_state_dict(state, strict=False)
model.eval()

eot = tok.encode(EOT)[0]
prompt = USER_TAG + 'Write a story about a fox who learns to share.' + EOT + ASSISTANT_TAG
ids = torch.tensor([tok.encode(prompt)], device='cuda')
out = model.generate(ids, max_new_tokens=200, temperature=0.8, top_p=0.95, eos_id=eot)
print(tok.decode(out[0].tolist()))

## 3. Build candidate preference pairs (~1 hour)

Samples 2 completions per held-out prompt at T=0.9, top-p=0.95. Heuristic-filters pairs that are too similar to provide DPO signal.

In [ ]:
!python scripts/build_preference_dataset.py \
    --sft_checkpoint checkpoints/sft/final.pt \
    --vocab          data/tinystories_v2_vocab.json \
    --merges         data/tinystories_v2_merges.txt \
    --output         data/pref_candidates.jsonl \
    --num_prompts 1000 --max_new_tokens 200 \
    --device cuda --dtype bfloat16

## 4. Label preferences with local Qwen judge (~1 hour)

Each pair is judged in BOTH orders. Pairs where the judge flips are dropped (position-bias control). Aim for ≥300 high-confidence pairs.

In [ ]:
!python scripts/label_preferences.py \
    --input  data/pref_candidates.jsonl \
    --output data/pref_labeled.jsonl \
    --judge_model Qwen/Qwen2.5-3B-Instruct \
    --device cuda --load_in_4bit

## 5. DPO training (~30 min A100, 400 steps)

Loads the SFT checkpoint twice (policy trainable, ref frozen). β=0.1.
With only 178 labeled pairs, 400 steps (~9 epochs) is enough — 1500 would
overfit. Watch `margin` — it must trend up over training.

In [9]:
# 178 labeled pairs / batch 4 = ~44 steps per epoch.
# 400 steps ≈ 9 epochs — enough signal, not enough to memorize.
# Watch the printed `margin` value — it should trend up during training.
!python scripts/train_dpo.py \
    --sft_checkpoint checkpoints/sft/final.pt \
    --preferences    data/pref_labeled.jsonl \
    --vocab          data/tinystories_v2_vocab.json \
    --merges         data/tinystories_v2_merges.txt \
    --checkpoint_dir checkpoints/dpo \
    --total_steps 400 --batch_size 4 \
    --lr_max 5e-6 --beta 0.1 \
    --device cuda --dtype bfloat16

Device: cuda, dtype: torch.bfloat16, beta: 0.1
Tokenizer vocab_size=16000, pad_id=256

Packing preferences from data/pref_labeled.jsonl ...
  178 preference pairs packed (max_length=512)

Loading policy and reference models ...
  policy/ref dtype: {torch.bfloat16}
step   10 | loss 0.6945 | margin +0.0099 | acc 45.00% | chosen_r -0.025 | rej_r +0.100 | lr 1.00e-06 | gn 103.00 | 2.3s
step   20 | loss 0.7047 | margin -0.0088 | acc 37.50% | chosen_r +0.100 | rej_r +0.000 | lr 2.00e-06 | gn 91.00 | 4.2s
step   30 | loss 0.6891 | margin +0.0210 | acc 45.00% | chosen_r +0.000 | rej_r -0.075 | lr 3.00e-06 | gn 93.50 | 6.1s
step   40 | loss 0.6922 | margin +0.0199 | acc 50.00% | chosen_r +0.000 | rej_r +0.100 | lr 4.00e-06 | gn 123.50 | 7.9s
step   50 | loss 0.6738 | margin +0.0479 | acc 50.00% | chosen_r -0.025 | rej_r -0.025 | lr 5.00e-06 | gn 103.00 | 9.8s
step   60 | loss 0.6879 | margin +0.0200 | acc 42.50% | chosen_r +0.050 | rej_r +0.000 | lr 4.99e-06 | gn 102.50 | 11.6s
step   70 | loss

## 6. Full eval matrix → results.md (~60-70 min with 7B judge)

In [10]:
# Eval uses Qwen-2.5-7B-Instruct (vs 3B for labeling) — bigger judge,
# less position bias, more trustworthy win-rate. Adds ~30 min vs the 3B.
!python scripts/eval_all.py \
    --base_checkpoint checkpoints/base_60m/final.pt \
    --sft_checkpoint  checkpoints/sft/final.pt \
    --dpo_checkpoint  checkpoints/dpo/final.pt \
    --val_data        data/tinystories_v2_tokens_val.npy \
    --vocab           data/tinystories_v2_vocab.json \
    --merges          data/tinystories_v2_merges.txt \
    --output          results.md \
    --num_eval_prompts 150 \
    --judge_model Qwen/Qwen2.5-7B-Instruct \
    --device cuda --load_in_4bit

Device: cuda, dtype: torch.bfloat16

=== 1. Held-out perplexity ===
Loading base ...
  base: loss 2.9166  ppl 18.48
Loading sft ...
  sft: loss 3.0322  ppl 20.74
Loading dpo ...
  dpo: loss 3.0953  ppl 22.09

=== 2. Sampling 150 completions per model ===
  Held-out prompts loaded: 150
  Generating from base ...
    25/150  (1.6 min)
    50/150  (3.2 min)
    75/150  (4.7 min)
    100/150  (6.4 min)
    125/150  (8.1 min)
    150/150  (9.7 min)
  Generating from sft ...
    25/150  (1.7 min)
    50/150  (3.4 min)
    75/150  (5.1 min)
    100/150  (6.8 min)
    125/150  (8.5 min)
    150/150  (10.2 min)
  Generating from dpo ...
    25/150  (1.7 min)
    50/150  (3.4 min)
    75/150  (5.1 min)
    100/150  (6.7 min)
    125/150  (8.5 min)
    150/150  (10.2 min)

=== 3. Judging pairs with Qwen/Qwen2.5-7B-Instruct ===
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 77.58it/s, Materializing param=model.norm.weight]                             

In [11]:
# Read the results back so you can see them in this session.
with open('results.md') as f:
    print(f.read())

# Copy results.md back to Drive for safekeeping.
!cp results.md {DRIVE_ROOT}/results.md
print('Saved to Drive:', f'{DRIVE_ROOT}/results.md')

# Post-Training Eval Results

## Held-out perplexity

| Checkpoint | Loss (nats) | Perplexity |
|---|---|---|
| base | 2.9166 | 18.48 |
| sft | 3.0322 | 20.74 |
| dpo | 3.0953 | 22.09 |

## Pairwise judge win-rate matrix

Judge: `Qwen/Qwen2.5-7B-Instruct`  ·  Prompts: 150  ·  Both orders run.

Each cell = P(row beats column), consistent (non-flipped) judgements only.

| ↓ vs → | base | sft | dpo |
|---|---|---|---|
| base | — | 12.7% | 12.7% |
| sft | 35.3% | — | 26.0% |
| dpo | 27.3% | 25.3% | — |

## Swap-consistency (position-bias control)

| Pair | Consistent | Flipped | Consistency rate |
|---|---|---|---|
| base vs sft | 72 | 78 | 48.0% ⚠️ (low signal) |
| base vs dpo | 60 | 90 | 40.0% ⚠️ (low signal) |
| sft vs dpo | 77 | 73 | 51.3% ⚠️ (low signal) |

## Headline

- SFT beats base in **35.3%** of pairs.
- DPO beats SFT in **25.3%** of pairs.
- DPO beats base in **27.3%** of pairs.

## Sample completions (first 3 prompts)

### Prompt 1

> Summary: Lily and Ben find a wallet with 

## Done.

Headlines to inspect in `results.md`:
- **DPO beats SFT in X% of pairs.** If X ≥ 58% on 150 prompts with swap-consistency ≥ 70%, you have a defensible resume bullet.
- If DPO win-rate is in the 50-55% noise band, the implementation is still demonstrably correct (test suite passes, reward margin trended up during training) — the eval just couldn't detect a small effect at this scale. Report the result honestly in the README.

Next: commit `results.md` to the repo and update the README with the headline numbers.

---

## Trimmed v2 Pipeline: SFT → DPO → Eval (~1.5-2 hr, A100 standard)

**Goal:** Test the SFT-undertraining hypothesis at minimum compute cost.

**What's trimmed vs full v2:**
- ~~Build 2,500 new pref candidates~~ — skipped, uses existing 178 v1 pairs
- ~~Re-label with Qwen-32B judge~~ — skipped, stays Qwen-7B (apples-to-apples eval)

**What's kept:**
- SFT v2 at 6,000 steps (~5 epochs) — the highest-impact fix
- DPO v2 re-derived from sft_v2 (coherent stack)
- Eval v2 with same Qwen-7B judge as v1 (fair comparison)

**Runtime: A100 STANDARD (not high-RAM).** Confirm below before running.

### Setup — confirm A100 standard and latest commit

In [4]:
%cd /content/char-level-transformer-lm
!git pull
!git log -1 --oneline          # confirm bbdd6f4 or later
!nvidia-smi | head -4          # should show A100 ~40960 MiB (not high-RAM)

/content/char-level-transformer-lm
Already up to date.
3cbf45c (HEAD -> main, origin/main, origin/HEAD) feat(notebook): replace full v2 with trimmed v2 pipeline (A100 standard, ~1.5-2 hr)
Wed May 27 13:46:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+


### Step 1 — SFT v2 retrain [A100, ~25-35 min]

6,000 steps ≈ 4.8 epochs over 40K examples (v1 was 2,000 steps ≈ 1.6 epochs).
Everything else unchanged (LR, batch size, dataset).

**Do NOT overwrite `checkpoints/sft/`.** Output goes to `checkpoints/sft_v2/`.

In [5]:
!python scripts/train_sft.py \
    --base_checkpoint checkpoints/base_60m/final.pt \
    --sft_data        data/tinystories_v2_sft.pt \
    --checkpoint_dir  checkpoints/sft_v2 \
    --total_steps 6000 --batch_size 32 \
    --lr_max 3e-5 --warmup_steps 200 \
    --device cuda --dtype bfloat16 --compile

Device: cuda
Loading SFT data: data/tinystories_v2_sft.pt
  Train examples: 39,200
  Val examples:   800
  Seq length:     511
Parameters: 53,261,440
Loading base weights: checkpoints/base_60m/final.pt
Compiling model with backend='inductor'
step    20 | loss 4.5094 | ppl 90.87 | lr 3.00e-06 | gn 6.59 | 55.9s
step    40 | loss 4.4234 | ppl 83.38 | lr 6.00e-06 | gn 5.19 | 58.0s
step    60 | loss 4.3523 | ppl 77.66 | lr 9.00e-06 | gn 4.94 | 60.1s
step    80 | loss 4.2523 | ppl 70.27 | lr 1.20e-05 | gn 4.53 | 62.1s
step   100 | loss 4.3398 | ppl 76.70 | lr 1.50e-05 | gn 4.28 | 64.2s
step   120 | loss 4.0867 | ppl 59.54 | lr 1.80e-05 | gn 4.75 | 66.3s
step   140 | loss 4.0484 | ppl 57.31 | lr 2.10e-05 | gn 4.41 | 68.4s
step   160 | loss 4.0078 | ppl 55.03 | lr 2.40e-05 | gn 4.22 | 70.5s
step   180 | loss 3.9367 | ppl 51.25 | lr 2.70e-05 | gn 4.31 | 72.6s
step   200 | loss 3.8234 | ppl 45.76 | lr 3.00e-05 | gn 4.22 | 74.6s
  [val] step 200 | loss 3.7523 | ppl 42.62
step   220 | loss 3.7695 

In [6]:
# Verify sft_v2 checkpoint.
import os
ckpt = 'checkpoints/sft_v2/final.pt'
assert os.path.exists(ckpt), f"MISSING: {ckpt}"
size_mb = os.path.getsize(ckpt) / 1e6
print(f"checkpoints/sft_v2/final.pt  {size_mb:.0f} MB")
assert size_mb > 200, f"Suspiciously small ({size_mb:.0f} MB) — may be corrupt"
print("OK — check final train loss above; should be lower than v1 SFT final loss")

checkpoints/sft_v2/final.pt  320 MB
OK — check final train loss above; should be lower than v1 SFT final loss


### Step 2 — DPO v2 from sft_v2 using existing v1 prefs [A100, ~20-30 min]

Uses `data/pref_labeled.jsonl` (the same 178 v1 pairs) — preference data is unchanged.
Trains from `sft_v2` so the policy and reference model are both from the better SFT.

600 steps: 178 pairs / batch 4 = 44 steps/epoch → ~13 epochs (vs v1 400 steps / ~9 epochs).
Slightly longer because the sharper SFT base can chase the preference further.

**Health check:** `reward_margin` must trend upward — target ≥ +0.3 by step 600, accuracy ≥ 80%.
If margin is flat or negative at step ~150, kill and report.

In [7]:
!python scripts/train_dpo.py \
    --sft_checkpoint checkpoints/sft_v2/final.pt \
    --preferences    data/pref_labeled.jsonl \
    --vocab  data/tinystories_v2_vocab.json \
    --merges data/tinystories_v2_merges.txt \
    --checkpoint_dir checkpoints/dpo_v2 \
    --total_steps 600 --batch_size 4 \
    --beta 0.1 --lr_max 5e-6 \
    --device cuda --dtype bfloat16

Device: cuda, dtype: torch.bfloat16, beta: 0.1
Tokenizer vocab_size=16000, pad_id=256

Packing preferences from data/pref_labeled.jsonl ...
  178 preference pairs packed (max_length=512)

Loading policy and reference models ...
  policy/ref dtype: {torch.bfloat16}
step   10 | loss 0.7086 | margin -0.0250 | acc 30.00% | chosen_r -0.050 | rej_r +0.050 | lr 1.00e-06 | gn 103.00 | 2.6s
step   20 | loss 0.6980 | margin +0.0025 | acc 35.00% | chosen_r +0.000 | rej_r +0.000 | lr 2.00e-06 | gn 100.50 | 4.3s
step   30 | loss 0.7066 | margin -0.0151 | acc 37.50% | chosen_r -0.125 | rej_r -0.025 | lr 3.00e-06 | gn 104.50 | 6.1s
step   40 | loss 0.7145 | margin -0.0274 | acc 30.00% | chosen_r +0.000 | rej_r -0.050 | lr 4.00e-06 | gn 108.00 | 7.8s
step   50 | loss 0.6852 | margin +0.0250 | acc 50.00% | chosen_r -0.150 | rej_r -0.125 | lr 5.00e-06 | gn 106.00 | 9.5s
step   60 | loss 0.6930 | margin +0.0137 | acc 40.00% | chosen_r -0.050 | rej_r +0.100 | lr 5.00e-06 | gn 116.50 | 11.3s
step   70 | lo

In [8]:
# Verify dpo_v2 checkpoint.
import os
ckpt = 'checkpoints/dpo_v2/final.pt'
assert os.path.exists(ckpt), f"MISSING: {ckpt}"
size_mb = os.path.getsize(ckpt) / 1e6
print(f"checkpoints/dpo_v2/final.pt  {size_mb:.0f} MB  OK")

checkpoints/dpo_v2/final.pt  320 MB  OK


### Step 3 — Eval v2 with Qwen-7B judge [A100, ~45-60 min]

Same judge as v1 (`Qwen2.5-7B-Instruct`) — apples-to-apples comparison.

Compare `results_v2.md` against `results.md` (v1):
- SFT-vs-Base win-rate should go **up** from 35.3% if sft_v2 is better trained.
- DPO-vs-SFT win-rate was essentially tied at 25.3% vs 26.0% — watch if that changes.
- swap-consistency should be similar (~40-51%) since judge is the same.

In [9]:
!python scripts/eval_all.py \
    --base_checkpoint checkpoints/base_60m/final.pt \
    --sft_checkpoint  checkpoints/sft_v2/final.pt \
    --dpo_checkpoint  checkpoints/dpo_v2/final.pt \
    --val_data        data/tinystories_v2_tokens_val.npy \
    --vocab           data/tinystories_v2_vocab.json \
    --merges          data/tinystories_v2_merges.txt \
    --judge_model     Qwen/Qwen2.5-7B-Instruct --load_in_4bit \
    --output          results_v2.md --device cuda

Device: cuda, dtype: torch.bfloat16

=== 1. Held-out perplexity ===
Loading base ...
  base: loss 2.8937  ppl 18.06
Loading sft ...
  sft: loss 3.0603  ppl 21.33
Loading dpo ...
  dpo: loss 3.0566  ppl 21.25

=== 2. Sampling 150 completions per model ===
TinyStories-Instruct-train.txt: 100% 2.66G/2.66G [00:06<00:00, 385MB/s]
TinyStories-Instruct-valid.txt: 100% 26.9M/26.9M [00:00<00:00, 62.7MB/s]
Generating train split: 100% 21755681/21755681 [00:23<00:00, 928051.77 examples/s]
Generating validation split: 100% 218380/218380 [00:00<00:00, 961076.57 examples/s]
  Held-out prompts loaded: 150
  Generating from base ...
    25/150  (1.5 min)
    50/150  (3.0 min)
    75/150  (4.3 min)
    100/150  (5.8 min)
    125/150  (7.4 min)
    150/150  (9.0 min)
  Generating from sft ...
    25/150  (1.6 min)
    50/150  (3.2 min)
    75/150  (4.8 min)
    100/150  (6.3 min)
    125/150  (8.0 min)
    150/150  (9.6 min)
  Generating from dpo ...
    25/150  (1.6 min)
    50/150  (3.1 min)
    75/15

In [10]:
# Print both result files side-by-side.
print("=" * 60 + "  results_v2.md (v2)  " + "=" * 60)
with open('results_v2.md') as f:
    print(f.read())

print("\n" + "=" * 60 + "  results.md (v1)  " + "=" * 60)
with open('results.md') as f:
    print(f.read())

============================================================  results_v2.md (v2)  ============================================================
# Post-Training Eval Results

## Held-out perplexity

| Checkpoint | Loss (nats) | Perplexity |
|---|---|---|
| base | 2.8937 | 18.06 |
| sft | 3.0603 | 21.33 |
| dpo | 3.0566 | 21.25 |

## Pairwise judge win-rate matrix

Judge: `Qwen/Qwen2.5-7B-Instruct`  ·  Prompts: 150  ·  Both orders run.

Each cell = P(row beats column), consistent (non-flipped) judgements only.

| ↓ vs → | base | sft | dpo |
|---|---|---|---|
| base | — | 10.0% | 10.0% |
| sft | 38.0% | — | 34.7% |
| dpo | 32.7% | 31.3% | — |

## Swap-consistency (position-bias control)

| Pair | Consistent | Flipped | Consistency rate |
|---|---|---|---|
| base vs sft | 72 | 78 | 48.0% ⚠️ (low signal) |
| base vs dpo | 64 | 86 | 42.7% ⚠️ (low signal) |
| sft vs dpo | 99 | 51 | 66.0% ⚠️ (low signal) |

## Headline

- SFT beats base in **38.0%** of pairs.
- DPO beats SFT in **31.3%** of pai

FileNotFoundError: [Errno 2] No such file or directory: 'results.md'

### Step 4 — Playground with v2 checkpoints [A100 or switch to L4]

Can switch to free L4/T4 here to save units — only 60M models, no big judge needed.

**Screenshot settings:**
- **Aligned tab**: prompt = `Write a short story about a girl named Lily who finds a magic stone in the forest.`  
  T=0.7, top-p=0.95, top-k=50, rep_penalty=1.15, max=220. Click "Generate from every checkpoint".  
  Screenshot → `docs/playground-aligned.png`
- **Generate tab**: prompt = `Once upon a time there was a little girl who`  
  T=0.8, top-p=0.95, top-k=50, min-p=0.05, rep_penalty=1.1, max=300.  
  Screenshot → `docs/playground-generate.png` (overwrites v1)
- Copy the 3 Aligned outputs to text for README.

In [ ]:
!python scripts/playground.py \
    --checkpoint     checkpoints/base_60m/final.pt \
    --checkpoint_sft checkpoints/sft_v2/final.pt \
    --checkpoint_dpo checkpoints/dpo_v2/final.pt \
    --vocab  data/tinystories_v2_vocab.json \
    --merges data/tinystories_v2_merges.txt \
    --dtype bfloat16 --share

In [4]:
!pip install -q bitsandbytes accelerate
!python -c "import bitsandbytes; print('bnb', bitsandbytes.__version__)"

bnb 0.49.2


In [6]:
!python scripts/eval_all.py \
    --base_checkpoint checkpoints/base_60m/final.pt \
    --sft_checkpoint  checkpoints/sft_v2/final.pt \
    --dpo_checkpoint  checkpoints/dpo_v2/final.pt \
    --val_data        data/tinystories_v2_tokens_val.npy \
    --vocab           data/tinystories_v2_vocab.json \
    --merges          data/tinystories_v2_merges.txt \
    --judge_model     Qwen/Qwen2.5-32B-Instruct --load_in_4bit \
    --num_eval_prompts 100 \
    --output          results_v2_qwen32b.md --device cuda

Device: cuda, dtype: torch.bfloat16

=== 1. Held-out perplexity ===
Loading base ...
  base: loss 2.8572  ppl 17.41
Loading sft ...
  sft: loss 3.0984  ppl 22.16
Loading dpo ...
  dpo: loss 3.0537  ppl 21.19

=== 2. Sampling 100 completions per model ===
  Held-out prompts loaded: 100
  Generating from base ...
    25/100  (1.5 min)
    50/100  (3.1 min)
    75/100  (4.5 min)
    100/100  (6.0 min)
  Generating from sft ...
    25/100  (1.6 min)
    50/100  (3.2 min)
    75/100  (4.9 min)
    100/100  (6.6 min)
  Generating from dpo ...
    25/100  (1.6 min)
    50/100  (3.3 min)
    75/100  (4.8 min)
    100/100  (6.5 min)

=== 3. Judging pairs with Qwen/Qwen2.5-32B-Instruct ===
Loading weights:  67% 514/771 [02:07<01:03,  4.03it/s, Materializing param=model.layers.42.self_attn.o_proj.weight]         
Traceback (most recent call last):
  File "/content/char-level-transformer-lm/scripts/eval_all.py", line 436, in <module>
    main()
  File "/content/char-level-transformer-lm/scripts/ev

In [3]:
!python scripts/eval_all.py \
      --base_checkpoint checkpoints/base_60m/final.pt \
      --sft_checkpoint  checkpoints/sft_v2/final.pt \
      --dpo_checkpoint  checkpoints/dpo_v2/final.pt \
      --val_data        data/tinystories_v2_tokens_val.npy \
      --vocab           data/tinystories_v2_vocab.json \
      --merges          data/tinystories_v2_merges.txt \
      --judge_model     Qwen/Qwen2.5-14B-Instruct \
      --num_eval_prompts 75 \
      --output          results_v2_qwen14b.md --device cuda

Device: cuda, dtype: torch.bfloat16

=== 1. Held-out perplexity ===
Loading base ...
  base: loss 2.8647  ppl 17.54
Loading sft ...
  sft: loss 3.0766  ppl 21.68
Loading dpo ...
  dpo: loss 3.1059  ppl 22.33

=== 2. Sampling 75 completions per model ===
  Held-out prompts loaded: 75
  Generating from base ...
    25/75  (1.7 min)
    50/75  (3.3 min)
    75/75  (4.8 min)
  Generating from sft ...
    25/75  (1.8 min)
    50/75  (3.5 min)
    75/75  (5.2 min)
  Generating from dpo ...
    25/75  (1.8 min)
    50/75  (3.5 min)
    75/75  (5.2 min)

=== 3. Judging pairs with Qwen/Qwen2.5-14B-Instruct ===
config.json: 100% 663/663 [00:00<00:00, 3.85MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 21.8MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 118MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 115MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 148MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 47.5k/47.5k [00:00<00:00, 113MB/

In [5]:
!cat results_v2_qwen14b.md


# Post-Training Eval Results

## Held-out perplexity

| Checkpoint | Loss (nats) | Perplexity |
|---|---|---|
| base | 2.8647 | 17.54 |
| sft | 3.0766 | 21.68 |
| dpo | 3.1059 | 22.33 |

## Pairwise judge win-rate matrix

Judge: `Qwen/Qwen2.5-14B-Instruct`  ·  Prompts: 75  ·  Both orders run.

Each cell = P(row beats column), consistent (non-flipped) judgements only.

| ↓ vs → | base | sft | dpo |
|---|---|---|---|
| base | — | 14.7% | 17.6% |
| sft | 24.0% | — | 12.0% |
| dpo | 31.1% | 24.0% | — |

## Swap-consistency (position-bias control)

| Pair | Consistent | Flipped | Consistency rate |
|---|---|---|---|
| base vs sft | 29 | 46 | 38.7% ⚠️ (low signal) |
| base vs dpo | 36 | 38 | 48.6% ⚠️ (low signal) |
| sft vs dpo | 27 | 48 | 36.0% ⚠️ (low signal) |

## Headline

- SFT beats base in **24.0%** of pairs.
- DPO beats SFT in **24.0%** of pairs.
- DPO beats base in **31.1%** of pairs.

## Sample completions (first 3 prompts)

### Prompt 1

> Summary: Lily and Ben find a wallet with 

In [ ]:
!python scripts/playground.py --checkpoint checkpoints/base_60m/final.pt --checkpoint_sft checkpoints/sft_v2/final.pt --checkpoint_dpo checkpoints/dpo_v2/final.pt --vocab data/tinystories_v2_vocab.json --merges data/tinystories_v2_merges.txt --dtype bfloat16 --share

[playground] Loaded BASE from checkpoints/base_60m/final.pt  (step 20,000)
[playground] Loaded SFT from checkpoints/sft_v2/final.pt  (step 6,000)
[playground] Loaded DPO from checkpoints/dpo_v2/final.pt  (step 600)
[playground] 53,261,440-param model on cuda  (3 checkpoint(s) loaded)
/content/char-level-transformer-lm/scripts/playground.py:315: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Transformer LM Playground", theme=theme) as app:
/content/char-level-transformer-lm/scripts/playground.py:373: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  stop_btn.click(fn=None, inputs=None, outputs=None, cancels=[gen_event])
* Running on local URL:  http://127.0.0.1:7860
* Run

### Report back — fill in after all steps complete

```
v2 SFT final loss: ___ (vs v1 ___)
v2 DPO reward_margin: +___ → +___ (vs v1 +0.01 → +0.34)
v2 DPO accuracy: ___% → ___% (vs v1 45% → 85%)

eval_v2 vs eval_v1:
- Base / SFT / DPO PPL:  ___ / ___ / ___   (was 18.48 / 20.74 / 22.09)
- SFT beats Base: ___%   (was 35.3%)
- DPO beats SFT:  ___%   (was 25.3%)
- DPO beats Base: ___%   (was 27.3%)
- swap-consistency: ___% (was 40-51%)

Qualitative from playground:
- Entity tracking improved? ___
- Repetition gone (with rep_penalty=1.15)? ___
- DPO still bland vs SFT? ___

Compute units used so far: ___ / 32
```